# Лабораторная работа №2. Логистическая и линейная регрессия

Фокин Леонид Александрович, группа 401, вариант 20.

## Данные и метрики

Используются Wholesale Customers (`Channel`, macro-F1 и accuracy) и SGEMM (время ядра в мс, MAE и RMSE). Тестовая доля 20%, пять фолдов для подбора, `random_state=20`. Подробнее о наборах: [data/README.md](data/README.md). Основные метрики подбора: macro-F1 вверх, MAE вниз.

Загружаю общие функции разбиения и оценки. В этой работе библиотечный бейзлайн использует стандартизацию признаков.

In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
import numpy as np
import pandas as pd
from IPython.display import display
from study import SEED, TASKS, evaluate, split, validate

BASE = 'scale'

def sklearn_model(task, params):
    if task == 'classification':
        return LogisticRegression(max_iter=2000, **params)
    return Ridge(**params) if params else LinearRegression()

## Бейзлайн sklearn

Фиксирую качество моделей со стандартными параметрами. Тестовые оценки не участвуют в выборе конфигурации.

In [ ]:
baseline = {task: evaluate(task, sklearn_model(task, {}), BASE) for task in TASKS}
display(pd.DataFrame([metrics for metrics, _ in baseline.values()]).round(4))

## Гипотезы и кросс-валидация

Стандартизация включена уже в бейзлайне. Логарифм трат и доли категорий могут упростить границу между каналами продаж. В регрессии кратности плитки и условные признаки кэша дают линейной модели дополнительные объясняющие переменные. Проверяю штраф L2 отдельно: меньший C означает более сильный штраф в классификации, больший alpha означает более сильный штраф в регрессии. При alpha > 0 библиотечная модель меняется с LinearRegression на Ridge.

В обоих случаях 5 фолдов. Сравниваю препроцессинг без подбора, подбор без изменения признаков и их комбинацию. Сначала выбираю лучшую альтернативу бейзлайну на CV, затем оцениваю её на отложенной части. `cv_std` показывает разброс оценок по фолдам, это не доверительный интервал.

In [ ]:
feature_options = ['features_scale']
class_grid = [{'C': 0.03}, {'C': 0.1}, {'C': 0.3}, {'C': 3}, {'C': 10}, {'C': 30}]
reg_grid = [{'alpha': 0.1}, {'alpha': 1}, {'alpha': 10}, {'alpha': 50}, {'alpha': 200}]
tables, chosen = {}, {}
for task, grid in zip(TASKS, (class_grid, reg_grid)):
    candidates = [('baseline', BASE, {})]
    candidates += [('preprocessing', prep, {}) for prep in feature_options]
    candidates += [('parameters', BASE, params) for params in grid]
    candidates += [('combined', prep, params) for prep in feature_options for params in grid]
    table = tables[task] = validate(task, lambda params: sklearn_model(task, params), candidates)
    chosen[task] = table.loc[table.experiment != 'baseline'].iloc[0]
    print(task)
    display(table.drop_duplicates('experiment').round(4).reset_index(drop=True))

Лучший вариант из альтернатив заново обучаю на всём train. Прирост на CV не гарантирует прироста на тесте.

In [ ]:
improved = {task: evaluate(task, sklearn_model(task, row.params), row.prep)
            for task, row in chosen.items()}
display(pd.DataFrame([metrics for metrics, _ in improved.values()]).round(4))